In [17]:
library(dplyr)
library(lubridate)

In [18]:
install.packages("googlesheets4")
library(googlesheets4)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [5]:
gs4_auth()

Is it OK to cache OAuth access credentials in the folder ~/.cache/gargle
between R sessions?
1: Yes
2: No


Selection: 1


Please point your browser to the following url: 

https://accounts.google.com/o/oauth2/v2/auth?client_id=603366585132-frjlouoa3s2ono25d2l9ukvhlsrlnr7k.apps.googleusercontent.com&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fspreadsheets%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email&redirect_uri=https%3A%2F%2Fwww.tidyverse.org%2Fgoogle-callback%2F&response_type=code&state=ce2edb3fe847edf652147357f5ac0786&access_type=offline&prompt=consent



Enter authorization code: eyJjb2RlIjoiNC8wQWZySWVwQld1MVA4UTd3X2NCc0lvUURESFNuM05CY3hkSUIwNk1zcG1xbWxpLUQ0YTNOVF91ck90TGtHeHEtdWNVM3hRUSIsInN0YXRlIjoiY2UyZWRiM2ZlODQ3ZWRmNjUyMTQ3MzU3ZjVhYzA3ODYifQ==


In [19]:
# Install and load necessary library
if(!require(readxl)) install.packages("readxl")
library(readxl)

In [20]:
# Replace with your actual URL
sheet_url <- "https://docs.google.com/spreadsheets/d/1if5EgzDmPfzjX1W6xPBXyjHBQ7erUzNTkCZGwtgYYCg/edit?gid=0#gid=0"

# Read the specific tab
df <- read_sheet(sheet_url, sheet = "Sheet1")

# Preview the data to ensure columns like 'Date', 'f1', and 'f2' are present
head(df)

✔ Reading from Project Dataset.

✔ Range ''Sheet1''.



date,f1,f2,f3
<dttm>,<dbl>,<dbl>,<dbl>
2016-06-01,-345.2,-343.3,0
2016-07-01,-356.7,-372.6,0
2016-08-01,-370.4,-409.0,0
2016-09-01,-370.0,-425.0,0
2016-10-01,-378.5,-437.3,0
2016-11-01,-390.3,-451.9,0


In [21]:
# Set persistence (rho) - using 0.9 as a standard high-persistence fiscal proxy
rho <- 0.9
sigma_sq <- 1 # This will cancel out later, so we set it to 1

# Build the 36x36 Omega matrix
times <- 1:36
Omega <- sigma_sq * (rho^abs(outer(times, times, "-")))

In [22]:
get_optimal_weights <- function(m, Omega) {
  # M Matrix (2 x 36): Row 1 for current year (f1), Row 2 for next year (f2), Row 3 for the year after next (f3)
  M <- matrix(0, 3, 36)
  M[1, 1:12] <- 1/12
  M[2, 13:24] <- 1/12
  M[3, 25:36] <- 1/12

  # N Vector (1 x 36): Target 12-month horizon starting at month m+1
  # e.g., if July (m=7), target is August (8) to next July (19)
  N <- rep(0, 36)
  N[(m + 1):(m + 12)] <- 1/12

  # Formula: W* = -(M %*% Omega %*% t(M))^-1 %*% (M %*% Omega %*% N)
  # Using t() for transpose and solve() for inverse
  W_star <- solve(M %*% Omega %*% t(M)) %*% (M %*% Omega %*% N)

  return(W_star)
}

In [23]:
# 1. Extract the month from your date column
# Assuming your date column is named 'date'
df <- df %>%
  mutate(month_num = month(date))

# 2. Apply the weight function to each row
# This creates a new column for the approximated 12-step-ahead forecast
df <- df %>%
  rowwise() %>%
  mutate(
    weights = list(get_optimal_weights(month_num, Omega)),
    w1 = weights[1],
    w2 = weights[2],
    w3 = weights[3],
    x_tilde = (w1 * f1) + (w2 * f2) + (w3 * f3)
  ) %>%
  ungroup()

In [24]:
# 3. Inspect the result
View(df[, c("date", "f1", "f2", "f3", "w1", "w2", "w3", "x_tilde")])

date,f1,f2,f3,w1,w2,w3,x_tilde
<dttm>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2016-06-01,-345.2,-343.3,0.0,5.370751e-01,0.60354035,-1.049257e-01,-392.59372
2016-07-01,-356.7,-372.6,0.0,4.344052e-01,0.70643309,-1.118869e-01,-418.16930
2016-08-01,-370.4,-409.0,0.0,3.328185e-01,0.79996223,-1.115250e-01,-450.46053
2016-09-01,-370.0,-425.0,0.0,2.354046e-01,0.88022974,-1.020796e-01,-461.19732
2016-10-01,-378.5,-437.3,0.0,1.452992e-01,0.94319022,-8.168949e-02,-467.45285
2016-11-01,-390.3,-451.9,0.0,6.571971e-02,0.98460600,-4.837180e-02,-470.59386
2016-12-01,0.0,-448.3,-417.8,9.714451e-17,1.00000000,1.110223e-16,-448.30000
2017-01-01,-446.2,-429.0,0.0,9.574557e-01,0.08011646,-1.721667e-02,-461.58670
2017-02-01,-431.4,-408.8,0.0,8.964965e-01,0.17353376,-3.659629e-02,-457.68920


In [25]:
# Replace with your actual Sheet URL or ID
target_sheet <- "https://docs.google.com/spreadsheets/d/1if5EgzDmPfzjX1W6xPBXyjHBQ7erUzNTkCZGwtgYYCg/edit?gid=0#gid=0"
sheet_write(df[, c("w1", "w2", "w3", "x_tilde")], ss = target_sheet, sheet = "E(fiscal balance)")

✔ Writing to Project Dataset.

✔ Writing to sheet 'E(fiscal balance)'.

